In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from collections import Counter
import time

In [11]:
headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/130.0.6723.58 Safari/537.36 (AirWatch Browser v21.09.0.9)'
        }


header_2 = {
    'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.8.1) Gecko/20061121 BonEcho/2.0'
}
url = 'https://empresite.eleconomista.es/Actividad/TECNOCUT-SL/'

header_3 = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.101 Safari/537.36'
}

header_4 = {
    'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.8.1.2pre) Gecko/20070213 BonEcho/2.0.0.2pre'
}


In [4]:
csv_entries = pd.read_csv('construction_entries_todo.csv', encoding ='latin1', names = ['company names'])

In [4]:
def import_csv(name: str):
    company_list = pd.read_csv(name+".csv")

In [5]:
def url_search_term(business_name: str):
    business_name = business_name.replace(",", "")
    business_name = business_name.replace(".", "")
    url_st = business_name.upper().replace(' ', '-')
    return url_st

In [12]:
r = requests.get(url,headers=header_4, timeout=10)
print(str(r.status_code))
soup = BeautifulSoup(r.content)

200


In [6]:
later_entries = csv_entries.iloc[300:,:]

In [7]:
def business_search(csv_entries: pd.DataFrame, n: float):
    """
    This function searches the baseurl site for the business names given in the DataFrame csv_entries.
    
    Parameters
    ----------
    """
    link_list = []
    name_match_guess = []
    baseurl = 'https://empresite.eleconomista.es/Actividad/'

    for i, company in enumerate(tqdm(list(csv_entries['company names']))):
        search_term = url_search_term(company)
        search_url = baseurl + search_term + '/'
        try:
            r = requests.get(search_url,headers=header_3, timeout=30)
            soup = BeautifulSoup(r.content)
            # print(company)
            # print(str(r.status_code))
            try:
                if str(r.status_code)== '200':
                    # first_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    # first_result = first_result.capitalize().split(' ')[0]
                    # company_name_start = company.split(' ')[0]
                    
                    # if first_result.split(' ')[0]==company.split(' ')[0]:
                    url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                    link_list.append(url_results)
                elif str(r.status_code)== '429':
                    print(f'current index is {i}')
                    break
                else:
                    link_list.append('possible 404 not found')
            except IndexError:
                link_list.append('no results found')
            time.sleep(n)
        except:
            link_list.append('requests issue')

    # links_found = list(set(link_list)).remove('404 not found')
    print(f'The cooldown this time was {n} seconds.')
    return link_list

In [8]:
link_list = business_search(later_entries, n = 2.5)

 14%|█▍        | 299/2119 [15:33<1:34:43,  3.12s/it]

current index is 299
The cooldown this time was 2.5 seconds.


In [51]:
Counter(link_list)

Counter({'429 issue': 1242,
         'requests issue': 917,
         '404 not found': 96,
         'https://empresite.eleconomista.es/BRICOINSA.html': 1,
         'https://empresite.eleconomista.es/MORRISON-INFRAESTRUCTURAS-CONSTRUCCIONES-SERVICIOS.html': 1,
         'https://empresite.eleconomista.es/SUEPRAT-IGUALADA.html': 1,
         'https://empresite.eleconomista.es/DECORACIONES-IMCASA.html': 1,
         'https://empresite.eleconomista.es/MATERIALS-CONSTRUCCIO-ILLA.html': 1,
         'https://empresite.eleconomista.es/COMERCIAL-SARIEGO.html': 1,
         'https://empresite.eleconomista.es/TRADINOX.html': 1,
         'https://empresite.eleconomista.es/DMC-CERAMICAS.html': 1,
         'https://empresite.eleconomista.es/UTREMEL-UTRERA.html': 1,
         'https://empresite.eleconomista.es/IMPERNOSA.html': 1,
         'https://empresite.eleconomista.es/ARICETA.html': 1,
         'https://empresite.eleconomista.es/CONSTRUCCIONES-JULIAN-FRANCO.html': 1,
         'https://empresite.elecon

In [44]:
def cleanup_found_links(df: pd.DataFrame, link_list: list) -> None:
    """
    This function adds urls to company listings that were found and removes rows of companies which were not found listed.

    Parameters
    ----------
    'df' : pd.DataFrame
        DataFrame of companies from csv import.
    'link_list' : list of links found from 'business_search' 
    """
    df = df.copy()
    df.reset_index(drop=True,inplace=True)
    df['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
    excluded_rows = df[(df['found_links']=='possible 404 not found') | (df['found_links']=='429 issue') | (df['found_links']=='requests issue')].index
    df.drop(index=excluded_rows, inplace=True)
    df = df.dropna(subset='found_links')
    return df

In [69]:
entries_infoadded = cleanup_found_links(later_entries, link_list)
entries_infoadded.reset_index(drop=True,inplace=True)

In [32]:
def get_contact_info(df: pd.DataFrame, n: int):
    """
    This company scrapes info of companies for which links were found.

    Parameters
    ----------
    'df' : pd.DataFrame
    """
    phonenumbers_found = []
    urls_found = []
    emails_found = []

    for link in tqdm(df['found_links']):
        r = requests.get(link,headers=header_4, timeout=10)
        if r.status_code != 200:
            print(str(r.status_code))
        soup = BeautifulSoup(r.content)
        try:
            found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
        except:
            found_url='not found'
        urls_found.append(found_url)
        try:
            found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
        except:
            found_email = 'not found'
        emails_found.append(found_email)
        try:
            found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
        except:
            found_phone = 'not found'
        phonenumbers_found.append(found_phone)
        time.sleep(n)

    info_dict = {'phones': phonenumbers_found, 'urls': urls_found, 'emails': emails_found}
    print(f'The cooldown this time was {n} seconds.')
    return info_dict

In [67]:
contact_info = get_contact_info(entries_infoadded, n=2.5)

  1%|          | 2/199 [00:05<09:42,  2.96s/it]

In [48]:
def add_found_info(df: pd.DataFrame, contact_info_dict: dict)-> None:
    """
    compiles phone numbers, emails, urls into df and fixes some formatting issues.

    Parameters
    ----------
    """
    phone_list = contact_info_dict['phones']
    url_list = contact_info_dict['urls']
    email_list = contact_info_dict['emails']
    
    df['phones'] = pd.DataFrame(phone_list, columns=['phones'])
    df['urls'] = pd.DataFrame(url_list, columns=['urls'])
    df['emails'] = pd.DataFrame(email_list, columns=['emails'])

    # Format corrections:
    df['urls'] = df['urls'].apply(lambda x: x[2:] if x.startswith('//')==True else x)
    df['emails'] = df['emails'].apply(lambda x: x.split(':')[1] if x.startswith('mailto')==True else x)
    df.drop(columns=['found_links'],inplace=True)
    
    return None

In [59]:
test = pd.DataFrame(contact_info['urls'], columns=['urls'])
test['urls'] = test['urls'].apply(lambda x: x[2:] if x.startswith('//')==True else x)

In [49]:
add_found_info(df=entries_infoadded, contact_info_dict=contact_info)

AttributeError: 'float' object has no attribute 'startswith'

In [ ]:
entries_infoadded.to_csv('some_processed_leads.csv')

In [155]:
r = requests.get(test_link,headers=header_3, timeout=10)
print(str(r.status_code))

soup = BeautifulSoup(r.content)

200


In [160]:
soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]

'mailto:cjfranco@iprodat.es'

In [164]:
soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text

'987238400'